# TextMamba3D — A100 ET Improvement Pipeline

**Goal:** Improve ET Dice from 79.10% → 82%+ via post-processing optimization + training fixes

| Phase | Cell | Description | GPU Time |
|-------|------|-------------|----------|
| Setup | 1-3 | Mount Drive, clone repo, extract data | ~2 min |
| **Phase 1** | **4** | **PP parameter sweep on val → test eval** | **~1 hour** |
| Phase 2 | 5-6 | V5.2a training (FTL+EE) + eval (only if Phase 1 < 82%) | ~11 hours |
| Compare | 7 | Results comparison table | instant |

In [ ]:
# Cell 1: Mount Drive + Install deps
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi 2>/dev/null || echo "No GPU"

# V5.2 uses Mamba-2 (not Mamba-3), simpler install
!pip install -q mamba-ssm causal-conv1d einops \
    transformers nibabel tensorboard pyyaml tqdm

In [ ]:
# Cell 2: Clone repo + extract data
import os, zipfile, shutil, subprocess, time

REPO_DIR = '/content/TextMamba3D'
DRIVE_BASE = '/content/drive/MyDrive/TextMamba3D'

git_dir = os.path.join(REPO_DIR, '.git')
if os.path.isdir(REPO_DIR) and not os.path.isdir(git_dir):
    print(f'Removing old non-git code at {REPO_DIR}...')
    shutil.rmtree(REPO_DIR)

if os.path.isdir(git_dir):
    os.chdir(REPO_DIR)
    subprocess.run(['git', 'pull'], check=True)
    print(f'Updated existing repo at {REPO_DIR}')
else:
    for attempt in range(1, 4):
        print(f'Cloning (attempt {attempt}/3)...')
        ret = subprocess.run(
            ['git', 'clone', '--depth', '1',
             'https://github.com/PlutoLei/TextMamba3D.git', REPO_DIR],
            capture_output=True, text=True
        )
        if ret.returncode == 0 and os.path.exists(
            os.path.join(REPO_DIR, 'models/textmamba3d.py')
        ):
            break
        print(f'  Failed (code {ret.returncode}): {ret.stderr.strip()}')
        if os.path.isdir(REPO_DIR):
            shutil.rmtree(REPO_DIR)
        if attempt < 3:
            time.sleep(5 * attempt)
    else:
        raise RuntimeError(
            f'Clone failed after 3 attempts. Last error: {ret.stderr.strip()}'
        )
    os.chdir(REPO_DIR)
    print(f'Cloned to {REPO_DIR}')

print(f'Working directory: {os.getcwd()}')

# Extract BraTS data
DATA_ZIP = os.path.join(DRIVE_BASE, "TextBraTS_data.zip")
DATA_DIR = os.path.join(
    REPO_DIR,
    "data/BraTS2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData"
)

if not os.path.exists(DATA_DIR):
    os.makedirs(os.path.dirname(DATA_DIR), exist_ok=True)
    if os.path.exists(DATA_ZIP):
        print(f"Extracting {DATA_ZIP}...")
        with zipfile.ZipFile(DATA_ZIP, 'r') as zf:
            zf.extractall(os.path.dirname(DATA_DIR))
        if os.path.exists(DATA_DIR):
            print(f"Data extracted. Cases: {len(os.listdir(DATA_DIR))}")
        else:
            print(f"ERROR: Expected path not found: {DATA_DIR}")
    else:
        print(f"ERROR: {DATA_ZIP} not found on Drive")
else:
    print(f"Data already exists. Cases: {len(os.listdir(DATA_DIR))}")

if os.path.exists(DATA_DIR):
    cases = [d for d in os.listdir(DATA_DIR)
             if os.path.isdir(os.path.join(DATA_DIR, d))]
    print(f"Total BraTS cases: {len(cases)}")

In [ ]:
# Cell 3: ET-enriched text restore
import os, sys, zipfile
os.chdir(REPO_DIR)

DATA_DIR = "./data/BraTS2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData"
ET_CACHE_ZIP = os.path.join(DRIVE_BASE, "et_enriched.zip")

cases = sorted(
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
) if os.path.isdir(DATA_DIR) else []
if not cases:
    raise RuntimeError(f"No BraTS cases found in {DATA_DIR}")

sample_enriched = os.path.join(
    DATA_DIR, cases[0], f"{cases[0]}_et_enriched.txt"
)

if os.path.exists(sample_enriched):
    count = sum(
        1 for d in cases
        if os.path.exists(os.path.join(DATA_DIR, d, f"{d}_et_enriched.txt"))
    )
    print(f"ET-enriched text already present for {count} cases, skipping")
elif os.path.exists(ET_CACHE_ZIP):
    print(f"Restoring ET-enriched text from {ET_CACHE_ZIP}...")
    with zipfile.ZipFile(ET_CACHE_ZIP, 'r') as zf:
        zf.extractall(DATA_DIR)
    count = sum(
        1 for d in cases
        if os.path.exists(os.path.join(DATA_DIR, d, f"{d}_et_enriched.txt"))
    )
    print(f"Restored ET-enriched text for {count} cases")
else:
    print("Generating ET-enriched text descriptions from T1ce images...")
    sys.path.insert(0, '.')
    from data.et_text_enrichment import process_all_cases
    results = process_all_cases(DATA_DIR)
    print(f"Generated for {len(results)} cases")
    with zipfile.ZipFile(ET_CACHE_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
        for case_dir in cases:
            et_file = os.path.join(
                DATA_DIR, case_dir, f"{case_dir}_et_enriched.txt"
            )
            if os.path.exists(et_file):
                zf.write(
                    et_file,
                    os.path.join(case_dir, f"{case_dir}_et_enriched.txt")
                )
    print(f"Cached ET text to {ET_CACHE_ZIP}")

et_count = sum(
    1 for d in cases
    if os.path.exists(os.path.join(DATA_DIR, d, f'{d}_et_enriched.txt'))
)
if et_count == 0:
    raise RuntimeError(
        'No ET-enriched text files found. Run text generation first.'
    )
print(f'ET-enriched text verified: {et_count}/{len(cases)} cases')

## Phase 1: Post-Processing Optimization (Zero Training Cost)

Sweep `--et-boost` / `--et-dilate` / `--et-min-size` on validation set, then evaluate best combination on test set.

- **et-boost**: Multiplies ET softmax channel before argmax (increases ET recall)
- **et-dilate**: Dilates ET mask within TC region (recovers boundary voxels)
- **et-min-size**: Minimum ET component size to keep (lower = keep more small ET)

All operations are TC/WT-safe: reclassifying class 1→3 does not change TC or WT.

In [ ]:
# Cell 4: Post-Processing Sweep on V5.0 checkpoint
# Sweep et_boost / et_dilate / et_min_size on VAL set, then eval best combo on TEST
import subprocess, os, re
from itertools import product as iterproduct
os.chdir(REPO_DIR)

DRIVE_CKPT = os.path.join(DRIVE_BASE, "checkpoints")
ckpt = os.path.join(DRIVE_CKPT, "best_v5.0.pth")
assert os.path.exists(ckpt), f"V5.0 checkpoint not found: {ckpt}"
CONFIG = "configs/textbrats_a100_v5.yaml"

# --- Sweep grid ---
ET_BOOSTS = [1.0, 1.1, 1.2, 1.5]
ET_DILATES = [0, 1]
ET_MIN_SIZES = [20, 50]

def run_eval(split, et_boost, et_dilate, et_min_size):
    cmd = [
        "python", "-u", "evaluate_full.py",
        "--config", CONFIG,
        "--checkpoint", ckpt,
        "--split", split,
        "--use-text", "--tta", "--postprocess",
        "--overlap", "0.5",
        "--et-boost", str(et_boost),
        "--et-dilate", str(et_dilate),
        "--et-min-size", str(et_min_size),
    ]
    ret = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True)
    if ret.returncode != 0:
        print(f"  FAILED: {ret.stderr[-200:]}")
        return None
    metrics = {}
    for line in ret.stdout.split("\n"):
        for key in ["dice_ET", "dice_TC", "dice_WT", "dice_mean"]:
            pat = key + r": ([\d.]+) \+/- ([\d.]+)"
            m = re.search(pat, line)
            if m:
                metrics[key] = float(m.group(1))
    return metrics

# --- Phase 1: Sweep on validation set ---
print("=" * 70)
print("POST-PROCESSING SWEEP (val set, text+TTA+PP)")
print("=" * 70)
results = []
for boost, dilate, min_sz in iterproduct(ET_BOOSTS, ET_DILATES, ET_MIN_SIZES):
    tag = f"boost={boost} dilate={dilate} min_sz={min_sz}"
    print(f"\n--- {tag} ---")
    m = run_eval("val", boost, dilate, min_sz)
    if m:
        results.append({"boost": boost, "dilate": dilate, "min_sz": min_sz, **m})
        et = m.get("dice_ET", 0)
        tc = m.get("dice_TC", 0)
        wt = m.get("dice_WT", 0)
        mn = m.get("dice_mean", 0)
        print(f"  ET={et:.4f}  TC={tc:.4f}  WT={wt:.4f}  Mean={mn:.4f}")

# --- Rank by ET Dice ---
print("\n" + "=" * 70)
print("RANKING (sorted by ET Dice)")
print("=" * 70)
results.sort(key=lambda r: r["dice_ET"], reverse=True)
header = f"{'Config':<35} {'ET':>8} {'TC':>8} {'WT':>8} {'Mean':>8}"
print(header)
print("-" * 70)
for r in results:
    tag = f"boost={r['boost']} dilate={r['dilate']} min={r['min_sz']}"
    print(f"{tag:<35} {r['dice_ET']:>7.4f} {r['dice_TC']:>7.4f} {r['dice_WT']:>7.4f} {r['dice_mean']:>7.4f}")

best = results[0]
print(f"\nBest: boost={best['boost']}, dilate={best['dilate']}, min_sz={best['min_sz']}")
print(f"  Val ET={best['dice_ET']:.4f}, TC={best['dice_TC']:.4f}, WT={best['dice_WT']:.4f}")

# --- Phase 2: Best combo on test set ---
print("\n" + "=" * 70)
print("FINAL TEST EVAL (best PP params from val sweep)")
print("=" * 70)
test_m = run_eval("test", best["boost"], best["dilate"], best["min_sz"])
if test_m:
    print(f"  TEST ET={test_m['dice_ET']:.4f}  TC={test_m['dice_TC']:.4f}  WT={test_m['dice_WT']:.4f}  Mean={test_m['dice_mean']:.4f}")
    print("\n--- V5.0 baseline (test, no PP optimization) ---")
    base_m = run_eval("test", 1.0, 0, 50)
    if base_m:
        print(f"  BASE ET={base_m['dice_ET']:.4f}  TC={base_m['dice_TC']:.4f}  WT={base_m['dice_WT']:.4f}  Mean={base_m['dice_mean']:.4f}")
        et_gain = test_m['dice_ET'] - base_m['dice_ET']
        print(f"\n  ET improvement: {et_gain:+.4f} ({et_gain*100:+.2f}%)")
        print(f"  TC change:      {test_m['dice_TC'] - base_m['dice_TC']:+.4f}")
        print(f"  WT change:      {test_m['dice_WT'] - base_m['dice_WT']:+.4f}")

## Phase 2: V5.2a Training — FTL + Edge Enhancement (Only if Phase 1 < 82%)

Fine-tune from V5.0 with fixed LR schedule + fresh optimizer:
- `--reset-lr`: Cosine schedule from epoch 0 (fixes V5.2b LR bug)
- `--reset-optimizer`: Fresh Adam moments (no stale DiceLoss gradients)
- `--es-metric weighted`: Early stopping monitors 0.5×ET + 0.25×TC + 0.25×WT

Per-class Dice (ET/TC/WT) logged every epoch. Abort if TC < 83%.

In [ ]:
# Cell 5: V5.2a Training (FTL + Edge Enhancement)
import subprocess, shutil, glob, os

os.chdir(REPO_DIR)
DRIVE_CKPT = os.path.join(DRIVE_BASE, "checkpoints")
V50_CKPT = os.path.join(DRIVE_CKPT, "best_v5.0.pth")
assert os.path.exists(V50_CKPT), f"V5.0 checkpoint not found: {V50_CKPT}"

# Clean local checkpoints
for f in glob.glob(os.path.join(REPO_DIR, "checkpoints/*.pth")):
    os.remove(f)
print("Cleaned local checkpoints for V5.2a fresh start")

def sync_checkpoints_to_drive(tag):
    """Sync local checkpoints to Drive with version tag."""
    local_ckpt = os.path.join(REPO_DIR, "checkpoints")
    if not os.path.exists(local_ckpt):
        return
    for f in glob.glob(os.path.join(local_ckpt, "*.pth")):
        dst = os.path.join(DRIVE_CKPT, os.path.basename(f))
        shutil.copy2(f, dst)
    # Save tagged best
    local_best = os.path.join(local_ckpt, "best.pth")
    if os.path.exists(local_best):
        tagged = os.path.join(DRIVE_CKPT, f"best_{tag}.pth")
        shutil.copy2(local_best, tagged)
        print(f"Best checkpoint saved as {tagged}")
    print(f"Synced checkpoints to {DRIVE_CKPT}")


# Train V5.2a with fixed LR schedule + fresh optimizer
ret = subprocess.run(
    ['python', '-u', 'train.py',
     '--config', 'configs/a100_v5.2a.yaml',
     '--resume', V50_CKPT,
     '--reset-lr',
     '--reset-optimizer',
     '--es-metric', 'weighted',
     '--no-text-ratio', '0.15',
     '--grad-accum', '2'],
    cwd=REPO_DIR,
)
if ret.returncode != 0:
    raise RuntimeError(
        f"V5.2a training failed with exit code {ret.returncode}"
    )

sync_checkpoints_to_drive("v5.2a")

# Archive logs
local_logs = os.path.join(REPO_DIR, "logs")
if os.path.isdir(local_logs):
    drive_logs = os.path.join(DRIVE_BASE, "logs_v5.2a")
    if os.path.isdir(drive_logs):
        shutil.rmtree(drive_logs)
    shutil.copytree(local_logs, drive_logs)
    print(f"Archived V5.2a logs to {drive_logs}")

print("V5.2a training complete!")

In [ ]:
# Cell 6: V5.2a 8-config Eval
import subprocess, os
os.chdir(REPO_DIR)

DRIVE_CKPT = os.path.join(DRIVE_BASE, "checkpoints")
ckpt = os.path.join(DRIVE_CKPT, "best_v5.2a.pth")
if not os.path.exists(ckpt):
    ckpt = os.path.join(REPO_DIR, "checkpoints/best.pth")
assert os.path.exists(ckpt), f"No V5.2a checkpoint found at {ckpt}"

CONFIG = 'configs/a100_v5.2a.yaml'

eval_configs = [
    ("text",          ["--use-text"]),
    ("text+PP",       ["--use-text", "--postprocess"]),
    ("text+TTA",      ["--use-text", "--tta"]),
    ("text+TTA+PP",   ["--use-text", "--tta", "--postprocess"]),
    ("notext",        ["--no-text"]),
    ("notext+PP",     ["--no-text", "--postprocess"]),
    ("notext+TTA",    ["--no-text", "--tta"]),
    ("notext+TTA+PP", ["--no-text", "--tta", "--postprocess"]),
]

print("=" * 70)
print("V5.2a EVALUATION (FTL + EE, fine-tuned from V5.0)")
print("=" * 70)

for name, flags in eval_configs:
    print(f"--- {name} ---")
    cmd = [
        'python', '-u', 'evaluate_full.py',
        '--config', CONFIG,
        '--checkpoint', ckpt,
        '--split', 'test',
        '--overlap', '0.5',
    ] + flags
    ret = subprocess.run(
        cmd, cwd=REPO_DIR, capture_output=True, text=True
    )
    print(ret.stdout[-500:] if len(ret.stdout) > 500 else ret.stdout)
    if ret.returncode != 0:
        print(f"WARNING: {name} eval failed: {ret.stderr[-300:]}")

print("V5.2a evaluation complete!")

## Results Comparison

In [ ]:
# Cell 7: Results comparison table + save to Drive
import json, os

DRIVE_EVAL = os.path.join(DRIVE_BASE, "eval_results")
os.makedirs(DRIVE_EVAL, exist_ok=True)

# V5.0 baseline (from previous eval)
v50 = {
    "text+TTA+PP": {"ET": 0.7910, "TC": 0.8560, "WT": 0.8967},
    "text+TTA":    {"ET": 0.7897, "TC": 0.8560, "WT": 0.8967},
    "text":        {"ET": 0.7760, "TC": 0.8402, "WT": 0.8866},
}

# Fill in from eval outputs above
v52b = {
    "text+TTA+PP": {"ET": 0.0, "TC": 0.0, "WT": 0.0},  # TODO: fill
    "text+TTA":    {"ET": 0.0, "TC": 0.0, "WT": 0.0},  # TODO: fill
    "text":        {"ET": 0.0, "TC": 0.0, "WT": 0.0},  # TODO: fill
}

v52a = {
    "text+TTA+PP": {"ET": 0.0, "TC": 0.0, "WT": 0.0},  # TODO: fill
    "text+TTA":    {"ET": 0.0, "TC": 0.0, "WT": 0.0},  # TODO: fill
    "text":        {"ET": 0.0, "TC": 0.0, "WT": 0.0},  # TODO: fill
}

print("=" * 70)
print("COMPARISON: V5.0 vs V5.2b (FTL) vs V5.2a (FTL+EE)")
print("=" * 70)
print(f"{'Config':<16} {'Version':<10} {'ET':>8} {'TC':>8} {'WT':>8} {'Mean':>8}")
print("-" * 60)

for cfg_name in ["text+TTA+PP", "text+TTA", "text"]:
    for ver_name, ver_data in [("V5.0", v50), ("V5.2b", v52b), ("V5.2a", v52a)]:
        d = ver_data[cfg_name]
        mean = (d["ET"] + d["TC"] + d["WT"]) / 3
        print(f"{cfg_name:<16} {ver_name:<10} "
              f"{d['ET']:>7.2%} {d['TC']:>7.2%} {d['WT']:>7.2%} {mean:>7.2%}")
    print()

# Ablation summary
print("ABLATION:")
for cfg in ["text+TTA+PP"]:
    ftl_delta = v52b[cfg]["ET"] - v50[cfg]["ET"]
    ee_delta = v52a[cfg]["ET"] - v52b[cfg]["ET"]
    print(f"  FTL contribution (V5.2b - V5.0):  ET {ftl_delta:+.2%}")
    print(f"  EE contribution  (V5.2a - V5.2b): ET {ee_delta:+.2%}")

# Save results
results = {"v5.0": v50, "v5.2b": v52b, "v5.2a": v52a}
with open(os.path.join(DRIVE_EVAL, "v5.2_results.json"), "w") as f:
    json.dump(results, f, indent=2)
print(f"Results saved to {DRIVE_EVAL}/v5.2_results.json")

## Resume Training (After Disconnect)

Run Cells 1-3 first to reinstall deps and restore data, then this cell.

In [ ]:
# Cell 8: Resume training after disconnect
# Run cells 1-3 first to reinstall deps and restore data, then this cell.
import os, shutil, glob

REPO_DIR = '/content/TextMamba3D'
DRIVE_BASE = '/content/drive/MyDrive/TextMamba3D'
DRIVE_CKPT = os.path.join(DRIVE_BASE, "checkpoints")
os.chdir(REPO_DIR)

# === CHANGE THESE for v5.2a vs v5.2b ===
RESUME_CONFIG = 'configs/a100_v5.2b.yaml'  # or 'configs/a100_v5.2a.yaml'
RESUME_TAG = 'v5.2b'                       # or 'v5.2a'

resume_ckpt = os.path.join(DRIVE_CKPT, "last.pth")

if os.path.exists(resume_ckpt):
    print(f"Resuming {RESUME_TAG} from {resume_ckpt}")
    !python -u train.py \
        --config {RESUME_CONFIG} \
        --resume "{resume_ckpt}" \
        --no-text-ratio 0.15 \
        --grad-accum 2

    # Sync checkpoints
    local_ckpt = os.path.join(REPO_DIR, "checkpoints")
    if os.path.exists(local_ckpt):
        for f in glob.glob(os.path.join(local_ckpt, "*.pth")):
            dst = os.path.join(DRIVE_CKPT, os.path.basename(f))
            shutil.copy2(f, dst)
        local_best = os.path.join(local_ckpt, "best.pth")
        if os.path.exists(local_best):
            tagged = os.path.join(DRIVE_CKPT, f"best_{RESUME_TAG}.pth")
            shutil.copy2(local_best, tagged)
            print(f"Tagged: {tagged}")
        print(f"Synced to {DRIVE_CKPT}")
else:
    print(f"No checkpoint to resume from at {resume_ckpt}")